In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install torch torchvision timm scikit-learn

Mounted at /content/drive


In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import torchvision.models as models
import numpy as np

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
train_transform = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(),
    T.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
train_dir = "/content/drive/MyDrive/RAF-DB/DATASET/train"
test_dir  = "/content/drive/MyDrive/RAF-DB/DATASET/test"

train_ds = ImageFolder(train_dir, transform=train_transform)
test_ds  = ImageFolder(test_dir, transform=test_transform)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)

In [ ]:
backbone = models.resnet50(pretrained=True)

num_features = backbone.fc.in_features

# Replace final classification layer
backbone.fc = nn.Sequential(
    nn.Linear(num_features, 512),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(512, 7)
)

model = backbone.to(device)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 165MB/s]


In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=7,
    gamma=0.1
)

In [ ]:
epochs = 25

for epoch in range(epochs):
    model.train()
    running_loss = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        outputs = model(x)
        loss = criterion(outputs, y)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    scheduler.step()

    print(f"Epoch [{epoch+1}/{epochs}] "
          f"Loss: {running_loss/len(train_loader):.4f}")

Epoch [1/25] Loss: 0.9800
Epoch [2/25] Loss: 0.6491
Epoch [3/25] Loss: 0.5426
Epoch [4/25] Loss: 0.4633
Epoch [5/25] Loss: 0.3916
Epoch [6/25] Loss: 0.3424
Epoch [7/25] Loss: 0.2975
Epoch [8/25] Loss: 0.1688
Epoch [9/25] Loss: 0.1165
Epoch [10/25] Loss: 0.0989
Epoch [11/25] Loss: 0.0838
Epoch [12/25] Loss: 0.0709
Epoch [13/25] Loss: 0.0567
Epoch [14/25] Loss: 0.0477
Epoch [15/25] Loss: 0.0397
Epoch [16/25] Loss: 0.0425
Epoch [17/25] Loss: 0.0423
Epoch [18/25] Loss: 0.0393
Epoch [19/25] Loss: 0.0356
Epoch [20/25] Loss: 0.0333
Epoch [21/25] Loss: 0.0322
Epoch [22/25] Loss: 0.0325
Epoch [23/25] Loss: 0.0352
Epoch [24/25] Loss: 0.0329
Epoch [25/25] Loss: 0.0315


In [ ]:
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        outputs = model(x)
        preds = outputs.argmax(1)

        y_true.extend(y.numpy())
        y_pred.extend(preds.cpu().numpy())

accuracy  = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average='weighted')
recall    = recall_score(y_true, y_pred, average='weighted')
f1        = f1_score(y_true, y_pred, average='weighted')

print("\n📊 ResEmoNet Performance on RAF-DB")
print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1 Score :", round(f1, 4))


📊 ResEmoNet Performance on RAF-DB
Accuracy : 0.8641
Precision: 0.8627
Recall   : 0.8641
F1 Score : 0.8631


In [ ]:
torch.save(
    model.state_dict(),
    "/content/drive/MyDrive/emotion_project/models/ResEmoNet.pth"
)

In [ ]:
import os
os.path.isfile("/content/drive/MyDrive/emotion_project/models/ResEmoNet.pth")

True